# Metrics, thresholds, and error analysis

**P0 Essential · D2 Independent · 90 minutes**

In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (average_precision_score, confusion_matrix, precision_score,
                             recall_score, roc_auc_score)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer

from pathlib import Path

def locate(relative: str, local_name: str | None = None) -> Path:
    candidates = []
    if local_name:
        candidates.append(Path.cwd() / local_name)
    candidates.extend(root / relative for root in [Path.cwd(), *Path.cwd().parents])
    for candidate in candidates:
        if candidate.exists():
            return candidate
    raise FileNotFoundError(
        f"Cannot find {relative}. Run from the course clone or place the downloaded data beside this notebook."
    )


In [ ]:
path = locate('datasets/teaching/predictive-maintenance/observations.csv', 'observations.csv')
data = pd.read_csv(path)
features = ['Type','Air temperature [K]','Process temperature [K]',
            'Rotational speed [rpm]','Torque [Nm]','Tool wear [min]']
X_train, X_test, y_train, y_test = train_test_split(
    data[features], data['Machine failure'], test_size=0.25, random_state=20260718,
    stratify=data['Machine failure'])
pre = ColumnTransformer([('type', OneHotEncoder(handle_unknown='ignore'), ['Type']),
                         ('num', StandardScaler(), features[1:])])
model = make_pipeline(pre, LogisticRegression(max_iter=1000, class_weight='balanced'))
model.fit(X_train, y_train)
scores = model.predict_proba(X_test)[:, 1]

## Task

Build a threshold report containing the confusion counts, precision, and recall. Keep the positive class and matrix orientation explicit.

In [ ]:
def threshold_report(y_true, scores, threshold: float) -> dict[str, float | int]:
    prediction = np.asarray(scores) >= threshold
    tn, fp, fn, tp = confusion_matrix(y_true, prediction, labels=[0, 1]).ravel()
    return {'threshold': threshold, 'tn': int(tn), 'fp': int(fp), 'fn': int(fn), 'tp': int(tp),
            'precision': precision_score(y_true, prediction, zero_division=0),
            'recall': recall_score(y_true, prediction, zero_division=0)}

In [ ]:
reports = pd.DataFrame([threshold_report(y_test, scores, t) for t in (0.3, 0.5, 0.7)])
assert (reports[['tn','fp','fn','tp']].sum(axis=1) == len(y_test)).all()
summary = {'roc_auc': roc_auc_score(y_test, scores),
           'pr_auc': average_precision_score(y_test, scores),
           'prevalence': float(y_test.mean())}
summary, reports

## Decision memo

Choose a threshold for high missed-failure cost and limited inspection capacity. Report expected false alarms and misses in this evaluation sample, one failure pattern to inspect, and why PR-AUC must be compared with prevalence. Do not describe the score as factory validation or causal evidence.

**Instructor note.** Accept more than one threshold when the consequence argument and counts agree. Penalise threshold selection on the final test, not a non-default threshold itself.